In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io.shapereader import natural_earth
from affine import Affine
from rasterio import features as rio_features
from scipy.ndimage import binary_erosion
from astropy.convolution import Gaussian2DKernel, interpolate_replace_nans

# === Pipeline test: fill → BMKG mask → NE overlay → potong NE 10m ===
# file_fvcom = '/data/share/output/netcdf/fvcom/2025/05/InaFlows_20250508_0000.nc'
file_fvcom = '/data/raid0/netcdf/inaCAWO/2026/11/InaCAWOWave_20261102_0000.nc'
file_mask = '/opendap/ww3gfs/land_mask_hires.nc'

lon_min, lon_max = 116, 126
lat_min, lat_max = -12, -6
var_name = 'hs'
vmin, vmax = 0, 3
output_png = 'cawowave_hs_fill_mask_ne10m_std1b.png'

# --- 1. Load FVCOM ---
ds_fvcom = xr.open_dataset(file_fvcom)
data_raw = ds_fvcom[var_name].isel(time=0).values.copy()
lon = ds_fvcom['lon'].values
lat = ds_fvcom['lat'].values
Lon, Lat = np.meshgrid(lon, lat)
ny, nx = len(lat), len(lon)

# --- 2. Gaussian fill (Astropy) ---
print('1/5 - Gaussian smoothing / fill NaN...')
kernel = Gaussian2DKernel(x_stddev=1)                   # Ganti nilai x_stddev untuk mengatur smoothing
data_filled = interpolate_replace_nans(data_raw, kernel)

# --- 3. Land mask BMKG hires → potong daratan ---
print('2/5 - Regrid land mask BMKG & potong daratan...')
ds_mask = xr.open_dataset(file_mask)
if 'longitude' in ds_mask.dims:
    ds_mask = ds_mask.rename({'longitude': 'lon', 'latitude': 'lat'})

# mask_ori: 0 = darat, 1 = laut
mask_regridded = ds_mask['mask_ori'].interp(
    lon=ds_fvcom['lon'], lat=ds_fvcom['lat'], method='nearest'
)
mask_values = mask_regridded.values

data_masked = data_filled.copy()
is_hires_land = mask_values == 0
is_outside_domain = np.isnan(mask_values)
is_orig_land = np.isnan(data_raw)
data_masked[is_hires_land] = np.nan
data_masked[is_outside_domain & is_orig_land] = np.nan

# Panel 4: data terpotong BMKG + overlay visual NE
data_bmkg_overlay = data_masked

# --- 4. Potong pakai Natural Earth 10m (rasterize ke grid FVCOM) ---
print('3/5 - Rasterize NE 10m land & potong data...')
ne_shp = natural_earth(resolution='10m', category='physical', name='land')
ne_land_gdf = gpd.read_file(ne_shp)
ne_land_gdf = ne_land_gdf.cx[lon.min():lon.max(), lat.min():lat.max()]

dx = float(lon[1] - lon[0])
dy = float(lat[1] - lat[0])
transform = Affine(dx, 0.0, lon[0] - dx / 2.0, 0.0, dy, lat[0] - dy / 2.0)
shapes = [(geom, 1) for geom in ne_land_gdf.geometry if geom is not None and not geom.is_empty]
ne_land_raster = rio_features.rasterize(
    shapes=shapes,
    out_shape=(ny, nx),
    transform=transform,
    fill=0,
    dtype=np.uint8,
)

# Crop lebih rileks: erosi mask darat N pixel agar tepi pantai tidak ikut terpotong
# → beberapa pixel laut tetap ada di bawah overlay daratan
land_erode_pixels = 1  # naikkan (2, 3, ...) agar makin longgar
is_ne_land = ne_land_raster == 1
if land_erode_pixels > 0:
    is_ne_land = binary_erosion(is_ne_land, iterations=land_erode_pixels)

data_ne_masked = data_filled.copy()
data_ne_masked[is_ne_land] = np.nan

print('4/5 - Menyusun plot 5 panel (2 kolom x 3 baris)...')
proj = ccrs.PlateCarree()
fig, axes = plt.subplots(
    3, 2,
    figsize=(14, 14),
    subplot_kw={'projection': proj},
)
axes_flat = axes.ravel()

panels = [
    (data_raw, '1. Mentah (InaCAWO Flows)'),
    (data_filled, '2. Gaussian Smoothing (GS)'),
    (data_masked, '3. GS Crop Land Mask BMKG'),
    (data_bmkg_overlay, '4. GS + Land Mask + overlay NE 10m'),
    (data_ne_masked, '5. GS + Crop NE 10m + overlay NE 10m'),
]

# NE 10m: semua panel land berwarna
ne_land_filled = cfeature.NaturalEarthFeature(
    category='physical',
    name='land',
    scale='10m',
    facecolor='none',
    edgecolor='#f54242',
    linewidth=0.4,
)

im = None
print('5/5 - Overlay Natural Earth 10m...')
for i, (data, title) in enumerate(panels):
    ax = axes_flat[i]
    im = ax.pcolormesh(
        Lon, Lat, data,
        transform=proj,
        shading='auto',
        cmap='viridis',
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False

    # Semua gambar: land diisi warna
    ax.add_feature(ne_land_filled, zorder=5)

# Slot ke-6 kosong (2 kolom x 3 baris, 5 panel)
axes_flat[5].set_visible(False)

# Colorbar vertikal di kanan (satu untuk semua panel)
fig.subplots_adjust(right=0.90, hspace=0.25, wspace=0.12)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.70])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical')
cbar.set_label(var_name, fontsize=11)

fig.savefig(output_png, dpi=300, bbox_inches='tight')
print(f'Selesai! PNG tersimpan: {output_png}')
plt.show()

ds_fvcom.close()
ds_mask.close()